# 🎯 LEGENDARY DATA PREPROCESSING (END-TO-END)
## Masterclass: Handling Messy Real-World Indian Datasets

### 🧠 THE MASTERPLAN
In this session, we will transform a raw, chaotic dataset into a "Machine Learning Ready" masterpiece. We'll be using a simulated but highly realistic **Indian Startup Funding Dataset**.

**Why Preprocessing?**
Imagine trying to cook a gourmet meal with unwashed, rotten vegetables. No matter how good your "algorithm" (recipe) is, the output will be terrible. In AI, this is called **GIGO: Garbage In, Garbage Out**.

**Sections covered:**
1. 🔹 Introduction & Loading Data
2. 🔹 Understanding Data Chaos
3. 🔹 Data Cleaning (Missing Values & Duplicates)
4. 🔹 Outlier Detection & Treatment (IQR & Z-Score)
5. 🔹 Categorical Encoding (Label, One-Hot, Target)
6. 🔹 Feature Scaling (Min-Max, Standardization)
7. 🔹 Power Transformations (Log, Box-Cox)
8. 🔹 Train-Test Split Strategies
9. 🔹 The Grand Finale: End-to-End Sklearn Pipelines

---
*Created for HERE AND NOW AI Masterclass*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from scipy import stats

# Set styling for graphics
sns.set_theme(style="whitegrid")
warnings.filterwarnings('ignore')

# --- 🧪 GENERATING A MESSY INDIAN DATASET ---
# Let's create a dataset simulating Indian Startup Funding with common issues
np.random.seed(42)
n_rows = 500

data = {
    'Startup_Name': [f'Startup_{i}' for i in range(n_rows)],
    'City': np.random.choice(['Bangalore', 'Mumbai', 'NCR', 'Delhi', 'Hyderabad', 'bangalore', 'MUMBAI', 'Pune', None], n_rows),
    'Investment_Type': np.random.choice(['Seed Funding', 'Series A', 'Series B', 'Private Equity', 'seed funding', None], n_rows),
    'Industry': np.random.choice(['FinTech', 'EdTech', 'SaaS', 'E-Commerce', 'Logistics', 'FoodTech', 'HealthTech', None], n_rows),
    'Amount_USD': np.random.exponential(scale=1000000, size=n_rows),
    'Stages_Complete': np.random.choice(['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4', 'Stage 5'], n_rows),
    'Founders_Experience': np.random.randint(2, 25, n_rows),
    'Date': pd.date_range(start='2020-01-01', periods=n_rows, freq='D')
}

df_raw = pd.DataFrame(data)

# Introduce chaotic mess:
# 1. Missing values
df_raw.loc[np.random.choice(n_rows, 50), 'Amount_USD'] = np.nan
df_raw.loc[np.random.choice(n_rows, 30), 'Founders_Experience'] = -99 # Placeholder for missing

# 2. Outliers
df_raw.loc[10:15, 'Amount_USD'] = [50000000, 75000000, 100000000, 120000000, 200000000, 45000000] # Unicorns

# 3. Duplicates
df_raw = pd.concat([df_raw, df_raw.iloc[0:10]], ignore_index=True)

# 4. Inconsistent Formatting (Amount as string with commas/symbols)
df_raw['Amount_USD_Messy'] = df_raw['Amount_USD'].apply(lambda x: f"₹ {x:,.0f}" if pd.notnull(x) else "nan")

print(f"Dataset generated with {df_raw.shape[0]} rows and {df_raw.shape[1]} columns.")
df_raw.head()

## 🔹 SECTION 2: INITIAL DATA INSPECTION & DIAGNOSTICS

### 🎯 Intuition
Before you fix something, you must understand where it's broken. This stage is like a **Clinical Checkup**—we're checking the "health" of the data.

### 🌎 Real World Impact
In high-stakes environments like Indian FinTech, a mismatch in decimal places or a missing "City" field can result in wrong financial risk assessments.

### ⚙️ How it Works
1. `.info()`: Tells us the "Data Types". Very often, numbers get misinterpreted as objects (strings).
2. `.describe()`: Shows statistical summaries. Look for "impossible" values (e.g., negative experience).
3. `.duplicated()`: Finds duplicate entries (common in real scraped data).

### 🚀 Implementation
We'll check our messy startup dataset.

In [ ]:
# 🏥 Perform Diagnostic Checks
print("--- 🩺 DATA INFO 🩺 ---")
print(df_raw.info())

print("\n--- 📊 SUMMARY STATISTICS 📊 ---")
print(df_raw.describe())

print("\n--- 🚫 MISSING VALUES 🚫 ---")
print(df_raw.isnull().sum())

print(f"\n--- 🎭 DUPLICATE ROWS: {df_raw.duplicated().sum()} 🎭 ---")

print("\n--- 📍 UNIQUE CITIES: ---")
print(df_raw['City'].unique()) # Notice: 'Bangalore' vs 'bangalore' vs 'NCR' (messy!)

## 🔹 SECTION 3: DATA CLEANING (MISSING VALUES & DUPLICATES)

### 🎯 Intuition
Missing values (NaNs) are like **holes in a bucket**. If we ignore them, the whole analysis leaks. Duplicates are **distorted echoes**—they make us think a trend is stronger than it really is.

### 🌎 Real World Impact
Imagine a Swiggy data point being counted twice—the model would think that particular area is twice as popular as it actually is, leading to biased resource allocation.

### ⚙️ Imputation Strategies
1. **Mean Imputation**: Good for symmetric, normally distributed data.
   - Formula: $\bar{x} = \frac{\sum{x_i}}{n}$
2. **Median Imputation**: Best for data with outliers (Skewed Data), like "Indian Funding Rounds" where a few startups get massive money.
   - Formula: Middle value when sorted.
3. **Mode Imputation**: Best for categorical values (e.g., most common "City").
4. **Dropping**: Removing the row/column. Use *ONLY* if the percentage of missing values is extremely low (<5%) and random.

### 🚀 Implementation
We'll clean up the missing investment amounts and fix the duplicated startups.

In [ ]:
# 🏥 Cleaning Strategy for Indian Startup Data
df_clean = df_raw.copy()

# 1. 🚫 DUPLICATES 🚫
print(f"Rows BEFORE dropping duplicates: {len(df_clean)}")
df_clean.drop_duplicates(inplace=True)
print(f"Rows AFTER dropping duplicates: {len(df_clean)}")

# 2. 📍 CLEANING CATEGORICAL DATA (City, Industry) 📍
# Notice: 'Bangalore' vs 'bangalore' vs 'MUMBAI'
df_clean['City'] = df_clean['City'].str.strip().str.title() # Strips space, capitalizes first letter
df_clean['City'] = df_clean['City'].fillna(df_clean['City'].mode()[0]) # Fill with Most Common City

# 3. 💸 CLEANING FUNDING (Numerical - Median Imputation) 💸
print(f"\nMissing values in Amount_USD before: {df_clean['Amount_USD'].isnull().sum()}")
funding_median = df_clean['Amount_USD'].median()
df_clean['Amount_USD'] = df_clean['Amount_USD'].fillna(funding_median)
print(f"Missing values in Amount_USD after: {df_clean['Amount_USD'].isnull().sum()}")

# 4. 🧠 HANDLING WRONG PLACEHOLDERS (Experience = -99) 🧠
# Treat -99 as missing
df_clean['Founders_Experience'] = df_clean['Founders_Experience'].replace(-99, np.nan)
# Mediam imputation for experience
df_clean['Founders_Experience'] = df_clean['Founders_Experience'].fillna(df_clean['Founders_Experience'].median())

df_clean.head()

## 🔹 SECTION 4: OUTLIER DETECTION & TREATMENT (IQR & Z-SCORE)

### 🎯 Intuition
Outliers are the **rebels** of your dataset—they don't follow the general trend. In Indian Startup Funding, a **Unicorn (>$1B)** is an "outlier" compared to a typical seed round.

### 🌎 Real World Impact
A massive outlier can "pull" the mean (average) towards it, making your model think *every* startup gets $10M, leading to huge business risks.

### ⚙️ Detection Methods
1. **IQR (Interquartile Range) Method**: Detect points beyond boundaries.
   - **Q1**: 25th percentile (Lower Quartile)
   - **Q3**: 75th percentile (Upper Quartile)
   - **IQR**: $Q3 - Q1$
   - **Lower Bound**: $Q1 - 1.5 \times IQR$
   - **Upper Bound**: $Q3 + 1.5 \times IQR$
2. **Z-Score Method**: Measures how many standard deviations a point is from the mean.
   - $Z = \frac{(X - \mu)}{\sigma}$
   - Outlier usually if $|Z| > 3$.

### 🚀 Implementation
Visualizing the outliers in funding amounts.

In [ ]:
# 🖼️ VISUALIZATION BEFORE TREATMENT 🖼️
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.boxplot(data=df_clean, x='Amount_USD')
plt.title('Distribution of Startup Funding (MESSY OUTLIERS!)')

# 1. IQR DETECTION #
Q1 = df_clean['Amount_USD'].quantile(0.25)
Q3 = df_clean['Amount_USD'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers_iqr = df_clean[(df_clean['Amount_USD'] < lower_bound) | (df_clean['Amount_USD'] > upper_bound)]
print(f"📊 IQR FOUND: {len(outliers_iqr)} outliers!")

# 2. IQR TREATMENT (Capping) #
print("\n--- 🔨 Capping Outliers (Winsorization) 🔨 ---")
df_clean['Amount_USD'] = np.where(df_clean['Amount_USD'] > upper_bound, upper_bound, df_clean['Amount_USD'])
df_clean['Amount_USD'] = np.where(df_clean['Amount_USD'] < lower_bound, lower_bound, df_clean['Amount_USD'])

# 🖼️ VISUALIZATION AFTER TREATMENT 🖼️
plt.subplot(1, 2, 2)
sns.boxplot(data=df_clean, x='Amount_USD')
plt.title('Distribution After Capping (CLEAN)')
plt.show()

## 🔹 SECTION 5: CATEGORICAL ENCODING (LABEL, ONE-HOT, ORDINAL, TARGET)

### 🎯 Intuition
ML models are **math engines**—they only understand numbers. They can't eat `Bangalore` or `SaaS`. We must translate words into numbers without losing meaning.

### 🌎 Real World Impact
In India's diverse market, an investor from "Mumbai" might have different preferences than "Delhi". Encoding these cities correctly allows a model to learn these local nuances.

### ⚙️ How it Works
1. **One-Hot Encoding**: Best for nominal (no order) data like `City`. It creates a column for each category.
2. **Label Encoding**: Assigns a number (0, 1, 2) to each. Use for **Target Variables** only (usually).
3. **Ordinal Encoding**: Use when there is an inherent **order** or **rank** (e.g., Stage 1 < Stage 2).
4. **Frequency Encoding**: Replacing the category with its frequency (e.g., if Mumbai appears 50 times, replace it with 50).
5. **Target Encoding**: Replaces category with the average of the target variable (e.g., average funding per city).

### 🚀 Implementation
We'll encode our categorical features.

In [ ]:
# 🏥 ENCODING STRATEGY 🏥
df_encoded = df_clean.copy()

# 1. 🥇 ONE-HOT ENCODING (for City - Nominal) 🥇
print(f"Shape BEFORE One-Hot Encoding: {df_encoded.shape}")
df_encoded = pd.get_dummies(df_encoded, columns=['City'], prefix='City', drop_first=True) # Avoiding dummy variable trap
print(f"Shape AFTER One-Hot Encoding: {df_encoded.shape}")

# 2. 🥈 ORDINAL ENCODING (for Stages - Natural Rank) 🥈
# Stage 1 < Stage 2 ...
stage_map = {'Stage 1': 1, 'Stage 2': 2, 'Stage 3': 3, 'Stage 4': 4, 'Stage 5': 5}
df_encoded['Stages_Encoded'] = df_encoded['Stages_Complete'].map(stage_map)

# 3. 🥉 FREQUENCY ENCODING (for Industry vertical) 🥉
# Very common when you have many categories
industry_freq = df_encoded['Industry'].value_counts().to_dict()
df_encoded['Industry_Freq'] = df_encoded['Industry'].map(industry_freq)

# 4. 🥇 TARGET ENCODING (for high-cardinality) 🥇
# (Simplified) Replace Industry with its average funding amount
industry_means = df_encoded.groupby('Industry')['Amount_USD'].mean().to_dict()
df_encoded['Industry_Target_Encoded'] = df_encoded['Industry'].map(industry_means)

df_encoded.head()

## 🔹 SECTION 6: FEATURE SCALING (MIN-MAX SCALING, STANDARDIZATION)

### 🎯 Intuition
Scaling is like **comparing apples to airplanes**. If you're comparing "Experience" (0-25 yrs) and "Funding" (0-$10M), the $10M will dominate your model's math, even if experience is more important.

### 🌎 Real World Impact
Distance-based algorithms like KNN, SVM, and Clustering *will* fail if you don't scale. A difference of 1 year of experience is tiny compared to a difference of ₹1,000,000 in salary in an Indian payroll dataset.

### ⚙️ How it Works
1. **Min-Max Scaling (Normalization)**: Rescales data to [0,1].
   - Formula: $X_{scaled} = \frac{(X - X_{min})}{(X_{max} - X_{min})}$
   - Best when you want to keep the data bounded.
2. **Standardization (Z-Score Scaling)**: Centers data around mean (0) and std dev (1).
   - Formula: $X_{scaled} = \frac{(X - \mu)}{\sigma}$
   - Resists outliers better than Min-Max.

### 🚀 Implementation
We'll scale our numerical columns.

In [ ]:
# 🏥 SCALING STRATEGY 🏥
df_scaled = df_encoded.copy()

# 1. 🥇 STANDARDIZATION (Scaling for Founders Experience) 🥇
scaler_std = StandardScaler()
df_scaled['Founders_Exp_Standardized'] = scaler_std.fit_transform(df_scaled[['Founders_Experience']])

# 2. 🥈 MIN-MAX SCALING (Scaling for Funding Amount) 🥈
scaler_minmax = MinMaxScaler()
df_scaled['Amount_USD_Normalized'] = scaler_minmax.fit_transform(df_scaled[['Amount_USD']])

# 3. 🥉 POWER TRANSFORMATIONS (For Right-Skewed Data) 🥉
# Log transformation is best for things that grow exponentially (like funding)
# (Visualizing skewness below)
df_scaled['Amount_USD_Log'] = np.log1p(df_scaled['Amount_USD']) # log(1+x)

# 🖼️ SHOWING EFFECTS 🖼️
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df_scaled['Amount_USD'], kde=True)
plt.title('Original Distribution (Heavily Skewed)')

plt.subplot(1, 2, 2)
sns.histplot(df_scaled['Amount_USD_Log'], kde=True)
plt.title('After Log Transformation (Closer to Normal)')
plt.show()

## 🔹 SECTION 8: TRAIN-TEST SPLIT (PREVENTING OVERFITTING)

### 🎯 Intuition
If you give an Indian board exam (like CBSE) by only memorizing previous year questions, you might not be able to solve a *new* problem. This is called **Overfitting**.

### 🌎 Real World Impact
In high-frequency trading (or identifying funded startups), a model that's overfitted will perform perfectly on old data but lose millions when faced with real-time market changes.

### ⚙️ How it Works
1. **Train Set**: Used to "Teach" the model.
2. **Test Set**: Used to "Test" the model on unseen data.
3. Common Split: 80% Train / 20% Test.

### 🚀 Implementation
Splitting our cleaned and encoded dataset.

In [ ]:
# 🏥 SPLITTING STRATEGY 🏥
X = df_scaled.drop(['Startup_Name', 'Amount_USD', 'Amount_USD_Messy', 'Stages_Complete', 'Industry', 'Date'], axis=1) # Features
y = df_scaled['Amount_USD_Log'] # Our Target (log-transformed funding)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total Rows: {len(df_scaled)}")
print(f"Training Rows: {len(X_train)}")
print(f"Testing Rows: {len(X_test)}")
X_train.head()

## 🔹 SECTION 9: THE GRAND FINALE: END-TO-END SKLEARN PIPELINE

### 🎯 Intuition
Doing all these steps manually *every single time* is **clunky, error-prone, and slow**. A **Pipeline** is like a factory assembly line. You put raw data in one side, and cleaned, scaled data comes out the other.

### 🌎 Real World Impact
In an industrial AI setting (e.g., predicting Uber ride prices in Mumbai), data flows in millions of rows per minute. A pipeline allows for **automated cleaning** without zero manual effort.

### ⚙️ How it Works
1. `SimpleImputer`: Handles missing values.
2. `StandardScaler`: Handles scaling.
3. `OneHotEncoder`: Handles categories.
4. `ColumnTransformer`: Directs each feature to its correct preprocessing "pathway".

### 🚀 Implementation
We'll build a single robust pipeline for our messy dataset.

In [ ]:
# 🏥 PIPELINE STRATEGY 🏥
# We'll use our initial df_raw (the messiest version)

# 1. DEFINE NUMERICAL PATH 📉
num_features = ['Founders_Experience']
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Handle missing values
    ('scaler', StandardScaler()) # Scaling
])

# 2. DEFINE CATEGORICAL PATH 🔡
cat_features = ['City', 'Industry']
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Handle missing cities
    ('encoder', OneHotEncoder(handle_unknown='ignore')) # OHE for nominal data
])

# 3. COMBINE UNDER COLUMN TRANSFORMER 🤖
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

# 4. FINAL FACTORY LINE 🏭
full_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Let's see the magic
X_raw = df_raw[['Founders_Experience', 'City', 'Industry']] # Features from the "MESSY" dataset
X_processed = full_pipeline.fit_transform(X_raw)

print(f"MESSY RAW DATA SHAPE: {X_raw.shape}")
print(f"CLEAN PROCESSED DATA SHAPE: {X_processed.shape} (after OHE expansion)")
print("\n--- ✅ DATA IS NOW ML-READY! ✅ ---")

## 🔹 SECTION 10: TRANSFORMED DATASET COMPARISON (BEFORE VS AFTER)

### 🏥 What we achieved:
1. **Dirty to Clean**: We fixed duplicate and missing values like experienced vs -99.
2. **Words to Numbers**: Our `City` and `Industry` are now 0s and 1s that models can read.
3. **Apples to Apples**: Extremely diverse ranges (1yr vs ₹10Cr) are now scaled and comparable.
4. **Resistant to Outliers**: We capped the "Unicorn" funding so they don't corrupt the analysis.

### 🎯 Conclusion
You are now ready to teach ANY Machine Learning algorithm how to predict startup success in India. **Preprocessing is 80% of the work—and you just mastered it.**

---
*Created for HERE AND NOW AI Masterclass*